# Φάση Γ: Classification Models

**Υπεύθυνος:** ML Engineer

**Μοντέλα:**
1. Random Forest
2. SVM


In [2]:
# Φάση Γ: Classification Models (Ισορροπημένες Παράμετροι & Ενσωμάτωση Βαρών)
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.types import DoubleType
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import (RandomForestClassifier, LinearSVC, NaiveBayes, 
                                       MultilayerPerceptronClassifier, GBTClassifier, LogisticRegression)

print("Εκκίνηση SparkSession...")
spark = SparkSession.builder \
    .appName("Stroke_Classification_Augmented") \
    .master("local[*]") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "8g") \
    .getOrCreate()

# 1. ΦΟΡΤΩΣΗ ΤΩΝ ΕΜΠΛΟΥΤΙΣΜΕΝΩΝ ΔΕΔΟΜΕΝΩΝ (Από το K-Means)
print("Φόρτωση δεδομένων με Cluster Assignments...")
train_gold = spark.read.parquet("../data/train_gold_with_cluster.parquet")
test_gold = spark.read.parquet("../data/test_gold_with_cluster.parquet")

train_gold = train_gold.withColumn("stroke", train_gold["stroke"].cast(DoubleType()))
test_gold = test_gold.withColumn("stroke", test_gold["stroke"].cast(DoubleType()))

# 2. FEATURE AUGMENTATION: Ενσωμάτωση του Cluster στα Features
print("Ενσωμάτωση του K-Means Cluster στο Feature Vector...")
train_gold = train_gold.withColumn("cluster", F.col("cluster").cast(DoubleType()))
test_gold = test_gold.withColumn("cluster", F.col("cluster").cast(DoubleType()))

assembler = VectorAssembler(inputCols=["features", "cluster"], outputCol="augmented_features")

train_gold = assembler.transform(train_gold).drop("features").withColumnRenamed("augmented_features", "features")
test_gold = assembler.transform(test_gold).drop("features").withColumnRenamed("augmented_features", "features")

train_gold.cache()
test_gold.cache()

# --- ADVANCED TECHNIQUE 1: Cost-Sensitive Learning (Βάρη Κλάσεων) ---
stroke_count = train_gold.filter(F.col("stroke") == 1.0).count()
total_count = train_gold.count()
balance_ratio = (total_count - stroke_count) / stroke_count
train_gold = train_gold.withColumn("class_weight", F.when(F.col("stroke") == 1.0, balance_ratio).otherwise(1.0))

# --- ADVANCED TECHNIQUE 2: Random Undersampling (Data-Centric Approach) ---
minority_df = train_gold.filter(F.col("stroke") == 1.0)
majority_df = train_gold.filter(F.col("stroke") == 0.0).sample(withReplacement=False, fraction=(stroke_count / (total_count - stroke_count)), seed=42)
train_undersampled = minority_df.union(majority_df)

# ==========================================
# 1. ΤΑ 4 ΑΡΧΙΚΑ ΣΟΥ ΜΟΝΤΕΛΑ
# ==========================================
print("Εκπαίδευση 4 Αρχικών Μοντέλων (Με χρήση Cluster Features)...")

# ΠΡΟΣΘΗΚΗ: weightCol="class_weight" ΚΑΙ maxDepth=5 για να μην κάνει overfit
rf = RandomForestClassifier(featuresCol="features", labelCol="stroke", weightCol="class_weight", numTrees=100, maxDepth=5, seed=22390225)
rf_preds = rf.fit(train_gold).transform(test_gold)

svm = LinearSVC(featuresCol="features", labelCol="stroke", maxIter=100)
svm_preds = svm.fit(train_gold).transform(test_gold)

nb = NaiveBayes(featuresCol="features", labelCol="stroke")
nb_preds = nb.fit(train_gold).transform(test_gold)

input_size = len(train_gold.select("features").first()[0])
# Επαναφορά σε μικρότερο δίκτυο
mlp = MultilayerPerceptronClassifier(maxIter=100, layers=[input_size, 16, 8, 2], seed=22390225, featuresCol="features", labelCol="stroke")
mlp_preds = mlp.fit(train_gold).transform(test_gold)


# ==========================================
# 2. ΤΑ ΝΕΑ ΜΟΝΤΕΛΑ (ΕΙΔΙΚΑ ΓΙΑ IMBALANCE)
# ==========================================
print("Εκπαίδευση Νέων Μοντέλων (GBT, Logistic Regression, Undersampled RF)...")

# Επαναφορά σε λιγότερα iterations
gbt = GBTClassifier(featuresCol="features", labelCol="stroke", maxIter=50, maxDepth=4, seed=22390225)
gbt_preds = gbt.fit(train_gold).transform(test_gold)

lr = LogisticRegression(featuresCol="features", labelCol="stroke", weightCol="class_weight", maxIter=100)
lr_preds = lr.fit(train_gold).transform(test_gold)

# Undersampled Random Forest
rf_under = RandomForestClassifier(featuresCol="features", labelCol="stroke", numTrees=100, maxDepth=5, seed=22390225)
rf_under_preds = rf_under.fit(train_undersampled).transform(test_gold)


# --- ΑΠΟΘΗΚΕΥΣΗ ΟΛΩΝ ---
print("Αποθήκευση προβλέψεων...")
rf_preds.select("stroke", "prediction", "probability").write.mode("overwrite").parquet("../data/rf_predictions.parquet")
svm_preds.select("stroke", "prediction", "rawPrediction").write.mode("overwrite").parquet("../data/svm_predictions.parquet")
nb_preds.select("stroke", "prediction", "probability").write.mode("overwrite").parquet("../data/nb_predictions.parquet")
mlp_preds.select("stroke", "prediction", "probability").write.mode("overwrite").parquet("../data/mlp_predictions.parquet")

gbt_preds.select("stroke", "prediction", "probability").write.mode("overwrite").parquet("../data/gbt_predictions.parquet")
lr_preds.select("stroke", "prediction", "probability").write.mode("overwrite").parquet("../data/lr_predictions.parquet")
rf_under_preds.select("stroke", "prediction", "probability").write.mode("overwrite").parquet("../data/rf_under_predictions.parquet")

spark.stop()
print("Η διαδικασία ολοκληρώθηκε επιτυχώς!")

Εκκίνηση SparkSession...
Φόρτωση δεδομένων με Cluster Assignments...
Ενσωμάτωση του K-Means Cluster στο Feature Vector...


26/06/09 18:15:29 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Εκπαίδευση 4 Αρχικών Μοντέλων (Με χρήση Cluster Features)...
Εκπαίδευση Νέων Μοντέλων (GBT, Logistic Regression, Undersampled RF)...
Αποθήκευση προβλέψεων...
Η διαδικασία ολοκληρώθηκε επιτυχώς!
